# COMM117 Week 7 Workshop — Evaluation Metrics + Summarisation (Extractive vs Abstractive)

**How to use:** Try the **TODO** cells first. Each TODO is followed by a **Model answer** cell.

**This week covers**
- Evaluation metrics: **semantic**, **human**, **system-level**
- Summarisation: **extractive** vs **abstractive**
- Reading link-back: **EAGLE** (speculative decoding extension) as a case study for *system-level evaluation*


## Part 0 — Setup (do not edit)

In [ ]:
# Part 0 — Setup (do not edit)

import math
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    return float(a @ b / denom) if denom > 0 else 0.0


## Part 1 — Semantic evaluation

### Exercise 1.1 — Why lexical metrics fail (concept)

**Prompt:** Give **one** example where two summaries mean the same thing but have low word overlap.
Then explain (2–3 lines) why **ROUGE** can score it poorly.


In [ ]:

example = "Ref: 'The meeting was postponed to Friday.'  Cand: 'They moved the meeting to Friday.'"
explanation = (
    "ROUGE mostly measures n-gram overlap, so paraphrases with different wording "
    "can have low overlap even if the meaning is the same."
)
example, explanation


### Exercise 1.2 — Implement ROUGE-L (Longest Common Subsequence)

ROUGE-L uses the **LCS** between reference and candidate token sequences.

Implement:
- `lcs_len(a, b)` → length of LCS
- `rouge_l_f1(ref, cand)` → F1 where:
  - precision = LCS / len(cand)
  - recall = LCS / len(ref)
  - F1 = 2PR/(P+R)

Use whitespace tokenization (`.split()`).


In [ ]:

def lcs_len(a: List[str], b: List[str]) -> int:
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n):
        for j in range(m):
            if a[i] == b[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])
    return dp[n][m]

def rouge_l_f1(ref: str, cand: str) -> float:
    rt = ref.split()
    ct = cand.split()
    if len(rt) == 0 or len(ct) == 0:
        return 0.0
    L = lcs_len(rt, ct)
    p = L / len(ct)
    r = L / len(rt)
    return 0.0 if (p + r) == 0 else 2*p*r/(p+r)

ref = "the cat sat on the mat"
cand = "the cat sat on mat"
rouge_l_f1(ref, cand)


### Exercise 1.3 — A simple semantic metric: TF‑IDF cosine similarity

We approximate semantic similarity by building TF‑IDF vectors and computing cosine similarity.

Complete `tfidf_cosine(ref, cand)`.


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_cosine(ref: str, cand: str) -> float:
    vec = TfidfVectorizer()
    X = vec.fit_transform([ref, cand]).toarray()
    return cosine(X[0], X[1])

ref = "The meeting was postponed to Friday."
cand = "They moved the meeting to Friday."
tfidf_cosine(ref, cand)


## Part 2 — Human evaluation

### Exercise 2.1 — Design a human evaluation rubric (summaries)

Create a 4-point rubric for **summary quality** with 3 criteria:
- Coverage
- Faithfulness
- Clarity

Write what **1 vs 4** looks like for each.


In [ ]:

rubric = {
    "coverage": {"1": "Misses most key points; focuses on minor details.",
                 "4": "Covers all major points; includes only important details."},
    "faithfulness": {"1": "Adds unsupported claims or contradicts the source.",
                      "4": "All statements supported by the source; no invented facts."},
    "clarity": {"1": "Hard to read; confusing structure.",
                "4": "Clear, concise, well-structured."},
}
rubric


### Exercise 2.2 — Inter-annotator agreement (Cohen’s Kappa)

Two tutors label 12 summaries as **Good (1)** or **Bad (0)**.
Compute Cohen’s kappa:

kappa = (p_o - p_e) / (1 - p_e)

- p_o = observed agreement
- p_e = chance agreement from marginals


In [ ]:

a = [1,1,0,1,0,0,1,1,0,1,0,1]
b = [1,0,0,1,0,0,1,1,0,0,0,1]

def cohens_kappa(a: List[int], b: List[int]) -> float:
    assert len(a) == len(b)
    n = len(a)
    po = sum(int(x==y) for x,y in zip(a,b)) / n
    pa1 = sum(a)/n
    pb1 = sum(b)/n
    pa0 = 1-pa1
    pb0 = 1-pb1
    pe = pa1*pb1 + pa0*pb0
    return 0.0 if (1-pe)==0 else (po-pe)/(1-pe)

cohens_kappa(a,b)


## Part 3 — System-level evaluation



### Exercise 3.1 — Compute throughput and latency 

You have inference with the senarial below.

- First Create a small set of inference logs (e.g., 10 requests) for a 60-second window.
  For each request, record:

  prompt_tokens (input length)

  output_tokens (generated length)

  ttft_ms (time until first token appears) : (first token − received) × 1000

  total_latency_ms (end-to-end time until the final token) : (completed − received) × 1000

- Then use your generated logs to compute:

  RPS = number of requests / 60

  output tokens/sec = sum(output_tokens) / 60

  p50 and p95 total latency

  average TTFT




=============

#### Scenario: 60-second LLM serving observation (timeline evidence)

You are monitoring a live LLM server for 60 seconds. For each request, the server records:

* Request received time (s)

* First token sent time (s)

* Response completed time (s)

* input_tokens (prompt length)

* output_tokens (generated length)

====== Evidence: 10 request records ========

1. received 0.000s, first token 0.220s, completed 1.450s, input_tokens=120, output_tokens=80

2. received 6.000s, first token 6.180s, completed 7.100s, input_tokens=80, output_tokens=60

3. received 12.000s, first token 12.310s, completed 14.200s, input_tokens=220, output_tokens=120

4. received 18.000s, first token 18.260s, completed 19.700s, input_tokens=150, output_tokens=90

5. received 24.000s, first token 24.140s, completed 24.800s, input_tokens=60, output_tokens=40

6. received 30.000s, first token 30.170s, completed 31.050s, input_tokens=90, output_tokens=55

7. received 36.000s, first token 36.240s, completed 37.500s, input_tokens=140, output_tokens=85

8. received 42.000s, first token 42.210s, completed 43.300s, input_tokens=110, output_tokens=70

9. received 48.000s, first token 48.300s, completed 50.100s, input_tokens=200, output_tokens=110

10. received 54.000s, first token 54.160s, completed 54.950s, input_tokens=75, output_tokens=50


In [ ]:

import pandas as pd
import numpy as np

logs = pd.DataFrame({
    "prompt_tokens":      [120,  80, 220, 150,  60,  90, 140, 110, 200,  75],
    "output_tokens":      [ 80,  60, 120,  90,  40,  55,  85,  70, 110,  50],
    "ttft_ms":            [220, 180, 310, 260, 140, 170, 240, 210, 300, 160],
    "total_latency_ms":   [1450,1100,2200,1700, 800,1050,1500,1300,2100, 950],
})

window_seconds = 60

def pctl(x, q):
    return float(np.percentile(np.array(x), q))

rps = len(logs) / window_seconds
out_toks_per_s = logs["output_tokens"].sum() / window_seconds
p50 = pctl(logs["total_latency_ms"], 50)
p95 = pctl(logs["total_latency_ms"], 95)
avg_ttft = float(logs["ttft_ms"].mean())

rps, out_toks_per_s, p50, p95, avg_ttft


### Exercise 3.2 — Why “system-level” matters (EAGLE link)

In 4–6 lines, explain why an inference speedup paper must report:
- hardware (GPU type)
- concurrency/batch settings
- prompt length distribution
- latency percentiles (p50/p95)


In [ ]:

answer = (
    "Speedups depend on system conditions: GPU/memory bandwidth, batch size/concurrency, and the mix of "
    "prompt/output lengths. Without these details results are not reproducible and comparisons can be unfair. "
    "Percentiles (especially p95) reveal tail-latency regressions that averages hide, which matters for real users."
)
answer


## Part 4 — Summarisation: Extractive vs Abstractive

### Exercise 4.1 — Extractive summarisation baseline (TF‑IDF sentence ranking)

Score each sentence by TF‑IDF cosine similarity to the whole document and select top-k sentences.
Complete `extractive_summary(text, k=2)`.


In [ ]:
# TODO
from sklearn.feature_extraction.text import TfidfVectorizer

doc = (
    "Large language models are expensive to serve. "
    "During inference, memory bandwidth is often the bottleneck because the KV cache grows with context length. "
    "vLLM improves throughput with PagedAttention, which reduces KV cache fragmentation. "
    "Speculative decoding can further reduce compute by letting a draft model propose tokens that a target model verifies in parallel. "
    "Decoding strategies like top-k and top-p control diversity during generation."
)




In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

def split_sents(text: str) -> List[str]:
    sents = [s.strip() for s in text.split(".") if s.strip()]
    return [s + "." for s in sents]

def extractive_summary(text: str, k: int = 2) -> List[str]:
    sents = split_sents(text)
    vec = TfidfVectorizer()
    X = vec.fit_transform(sents + [text]).toarray()
    doc_vec = X[-1]
    sent_vecs = X[:-1]
    scores = [cosine(v, doc_vec) for v in sent_vecs]
    top_idx = list(np.argsort(scores)[::-1][:k])
    return [sents[i] for i in top_idx]

extractive_summary(doc, k=2)


## Part 5 — Quick concept checks (5 marks)

1) (1) Why can ROUGE undervalue good paraphrases?
2) (1) What does Cohen’s kappa measure?
3) (1) Name two system-level metrics used in LLM serving.
4) (1) Why is p95 latency important?
5) (1) Give one hallucination risk unique to abstractive summarisation.


In [ ]:

a1 = "ROUGE is n-gram overlap; paraphrases can have low overlap with same meaning."
a2 = "Agreement between human raters beyond chance."
a3 = "Latency (TTFT, p50/p95), throughput (tokens/sec), RPS, GPU memory usage."
a4 = "It captures tail latency (worst-case user experience), not just averages."
a5 = "Generated summaries can invent facts not in the source (hallucination)."
[a1, a2, a3, a4, a5]
